# 🧠 Mathematical Foundations: White-Box Backpropagation

In autonomous robotics, deploying perception models to resource-constrained edge devices requires optimizing matrix operations directly in C++ or CUDA. Relying purely on high-level APIs like PyTorch is insufficient. 

This notebook demonstrates a profound understanding of the underlying matrix calculus by rigorously deriving the gradient of a linear layer and validating it against PyTorch's internal Autograd engine.

## 1. Forward Propagation
Given an input matrix $X \in \mathbb{R}^{1 \times 3}$ and a weight matrix $W \in \mathbb{R}^{3 \times 2}$, the prediction is defined as:
$$Y_{pred} = XW$$

## 2. Loss Function (MSE)
We use Mean Squared Error to measure the deviation from the target $Y_{target}$:
$$L = \frac{1}{2} \|XW - Y_{target}\|_F^2$$

## 3. Backward Propagation (Matrix Calculus)
To minimize the loss, we must compute the gradient of $L$ with respect to $W$. Applying the chain rule in multivariable calculus, we derive the exact gradient matrix:
$$\frac{\partial L}{\partial W} = X^T (Y_{pred} - Y_{target})$$

In [2]:
import torch
import numpy as np

# 1. Initialize matrices (using float64 for absolute mathematical precision)
X = torch.randn(1, 3, dtype=torch.float64)
W = torch.randn(3, 2, requires_grad=True, dtype=torch.float64)
Y_target = torch.randn(1, 2, dtype=torch.float64)

# 2. Forward Pass
Y_pred = X @ W
loss = 0.5 * torch.sum((Y_pred - Y_target)**2)

# ==========================================
# Engine A: PyTorch Automatic Differentiation
# ==========================================
loss.backward()
pytorch_grad = W.grad.clone().numpy()

# ==========================================
# Engine B: Pure Mathematical Derivation
# ==========================================
# Formula derived above: dL/dW = X^T @ (Y_pred - Y_target)
manual_grad = (X.t() @ (Y_pred - Y_target)).detach().numpy()

# 3. Precision Validation
difference = np.max(np.abs(pytorch_grad - manual_grad))
print("--- Gradient Validation ---")
print(f"Maximum absolute difference: {difference}")

if difference < 1e-10:
    print("✅ SUCCESS: The manual matrix calculus derivation perfectly matches PyTorch's internal C++ Autograd engine.")

--- Gradient Validation ---
Maximum absolute difference: 0.0
✅ SUCCESS: The manual matrix calculus derivation perfectly matches PyTorch's internal C++ Autograd engine.
